# 00 Task-Switch Stress Test

This notebook visualizes the main experiment results.

- Input: `result/table/00_task_switch_timeseries.csv`
- Output: PDF vector figures in `result/figure/`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
ts_path = ROOT / "result" / "table" / "00_task_switch_timeseries.csv"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ts_path)
df.head()


In [ ]:
# Average across runs for each method at each slot
g = df.groupby(["method", "t"], as_index=False).agg(
    alpha=("alpha", "mean"),
    sum_queue=("sum_queue", "mean"),
    sense_u=("sense_u", "mean"),
    sum_rate=("sum_rate", "mean"),
    noise_sense=("noise_sense", "mean"),
)
methods = list(g["method"].unique())
methods


In [ ]:
def plot_time_series(metric: str, ylabel: str, fname: str):
    plt.figure()
    for m in methods:
        sub = g[g["method"] == m]
        plt.plot(sub["t"], sub[metric], label=m)
    plt.xlabel("Slot t")
    plt.ylabel(ylabel)
    plt.legend()
    out = fig_dir / fname
    plt.tight_layout()
    plt.savefig(out, format="pdf")
    plt.close()
    print("Saved:", out)

plot_time_series("alpha", "Gate $\\alpha_t$", "00_alpha.pdf")
plot_time_series("sum_queue", "Sum Queue", "00_sum_queue.pdf")
plot_time_series("sense_u", "Sensing Uncertainty $u_t$", "00_sense_u.pdf")


In [ ]:
# Pareto-style scatter: mean sum_queue vs mean sense_u per method
pareto = df.groupby(["method"], as_index=False).agg(
    sum_queue=("sum_queue", "mean"),
    sense_u=("sense_u", "mean"),
)

plt.figure()
plt.scatter(pareto["sum_queue"], pareto["sense_u"])
for _, row in pareto.iterrows():
    plt.text(row["sum_queue"], row["sense_u"], row["method"], fontsize=8)
plt.xlabel("Mean Sum Queue (lower is better)")
plt.ylabel("Mean Sensing Uncertainty (lower is better)")
plt.tight_layout()
out = fig_dir / "00_pareto.pdf"
plt.savefig(out, format="pdf")
plt.close()
print("Saved:", out)
